### 1. Смоделируем реальную задачу: высокочастотный датчик вибрации на роторе турбины.

В таких данных простые средние значения (rolling.mean) или IsolationForest бессильны. Нормальная работа турбины — это не плоская линия, это синусоида (колесо крутится, вибрация идет волной).

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

np.random.seed(42)

# 1. Генериурем сигнал 5000 точек, от 0.0 до 50.0 секунд, Шаг (~0.01 сек)
# Частота дискретизации — равна 100 Гц (100 измерений в секунду).
t = np.linspace(0, 50, 5000)

# Основная синусоида (идеального круга не существует)
# Один полный цикл синусоиды (от пика до пика) — это ровно один оборот вала турбины.
rotation = np.sin(2 * np.pi * 1 * t)

# Шум (электромагнитные помехи, трение, потоки пара/газа)
# Нормальное распределение (или распределение Гаусса) - симметричный колокол
noise = np.random.normal(0, 0.2, 5000)

# Сигналы вибрации
vibration = rotation + noise

# моделируется работа системы диагностики турбины
y_true = np.zeros(5000) # 0 - норма, в эту мс турбина работает исправно

# Ложная аномалия (шум связи, случ. удар) - точка 1500
vibration[1500] += 5.0

# Реальная поломка: последние 500 точек появляется высокочастотная вибрация
vibration[-500:] += 0.8 * np.sin(10 * np.pi * t[-500:])
y_true[-500:] = 1

df = pd.DataFrame({"Vibration": vibration, "True_Label": y_true})

df.head()

,Vibration,True_Label
0,0.099343,0.0
1,0.035150,0.0
2,0.254896,0.0
3,0.492024,0.0
4,0.201908,0.0


---
#### 2. Подготовка окон

**Скользящее окно**: есть буфер (очередь) на 50 точек. Пришла 51-я точка: старая 1-я точка вылетает из буфера, новая залетает. Нейросеть мгновенно проверяет этот свежий срез (точки 2–51). Пришла 52-я точка: точки 2–51 превращаются в 3–52. Сеть снова мгновенно проверяет их. Скользящее окно в коде обучения нужно для того, чтобы смоделировать этот непрерывный поток. Мы учим нейросеть принимать решение на любом произвольном отрезке из 50 точек, который в будущем может прилететь из турбины в любую миллисекунду.
(window_size = 50, чтобы захватить период синусоиды), получаем 4951 * 50 = 247 550 чисел

`strides` - в оперативной памяти по-прежнему лежит всего один длинный исходный массив из 5000 чисел, NumPy просто создаёт «умную виртуальную карту» (изменяет шаг перемещения по памяти)

>Почему скользящего нет по дефолту в моделе обучения? (batch=50, windows=true)

В PyTorch и других библиотеках глубокого обучения строго разделены две сущности (разделение обязанностей):

1. **Модель (Нейросеть)** — её задача исключительно математическая: взять матрицу тензоров, перемножить на веса и выдать ответ. Она вообще «не знает», откуда взялись данные — из датчика турбины, из пикселей картинки или из текста.  

2. **Загрузчик данных (DataLoader)** — его задача подготовить эти самые матрицы.

Если бы нейросеть сама на ходу пыталась нарезать окна внутри себя, её код превратился бы в кашу. Архитектура требует: на вход модели должна подаваться уже готовая «коробка» с данными строго определенной формы.  

Делать нарезку окон **заранее вручную** — это учебный, упрощенный вариант. Для 5000 точек это нормально. Но если бы было 5 000 000 000 точек, ручной метод `create_windows` забил бы всю память, даже несмотря на оптимизации NumPy.

На реальном производстве используют инструмент PyTorch — **`Dataset`** и **`DataLoader`**.

>Пример Dataset/DataLoader в проде

In [ ]:
from torch.utils.data import Dataset, DataLoader

class TurbineDataset(Dataset):
    def __init__(self, vibration, window_size):
        self.vibration = vibration
        self.window_size = window_size

    def __len__(self):
        # Говорим загрузчику, сколько окон у нас *теоретически* есть
        return len(self.vibration) - self.window_size + 1

    def __getitem__(self, idx):
        # НА ХОДУ, ровно в момент обучения, вырезаем нужные 50 точек!
        window = self.vibration[idx : idx + self.window_size]
        return torch.FloatTensor(window)

# Вот он, аналог batch=50, где данные генерируются прямо в процессе обучения!
dataset = TurbineDataset(scaled_vibration, window_size=50)
loader = DataLoader(dataset, batch_size=32, shuffle=True) 

---

> Наш учебный вариант через самописную create_windows

In [ ]:
# Вызываем конструктор класса и создаем в оперативной памяти конкретный «физический» экземпляр (выделенную структуру данных)
# Методы: .fit(), .transform(), .fit_transform()
# Атрибуты: mean_, scale_, var_
scaler = StandardScaler()

# scaler.mean_ — сюда записывается массив с вычисленным средним значением
# scaler.scale_ — сюда записывается массив со стандартным отклонением (разбросом).
# scaler.n_samples_seen_ — количество строк, которые он прочитал (2000).
scaler.fit(df[["Vibration"]].iloc[:2000])

# z = (x - mean_ / scale_) - новое отмасштабированное значение
# Стандартизированная копия df["Vibration"] в numpy arr
scaled_vibration = scaler.transform(df[["Vibration"]])
print(type(scaled_vibration))
print(scaled_vibration.shape) # Колонка: 1 (Вибрация), Строки: 5000 (Значения)
print(scaled_vibration[:5])   # Аналог head()
print(scaled_vibration[1500]) # По индексу

window_size = 50

def create_windows(data, window_size):
    windows = [data[i:(i + window_size)] for i in range(len(data) - window_size + 1)]
    return np.array(windows)

X_windows = create_windows(scaled_vibration, window_size) # (4951, 50, 1)

<class 'numpy.ndarray'>
(5000, 1)
[[0.11811719]
 [0.03178823]
 [0.32731152]
 [0.64621188]
 [0.25605105]]
[6.94343617]


#### 2.1 **StandardScaler**
- делает масштаб «похожим на колокол» распределения Гаусса. Центр равен 0, базовый шаг равен 1, а аномалии могут принимать любые гигантские значения (10, 30, 100), выделяясь на общем фоне.

---

#### 3. Тензор

- это аналог numpyarr но для GPU и вычисления градиентов нейросети

Массивы NumPy умеют считаться только на центральном процессоре компьютерной системы (CPU) и не умеют автоматически вычислять градиенты для обучения моделей в отличии от тензоров и PyTorch

#### Типы тензоров

- **FloatTensor** float32 (32 бита), обеспечивает идеальный баланс между точностью и скоростью расчетов, стандарт для сигналов, картинок, весов.

- **DoubleTensor** float64 (64 бита), сверхточный (двойная точность). Нужен для сложных научных или космических расчетов. Жрет в 2 раза больше памяти и работает медленнее.

- **HalfTensor** float16/bfloat16 (16 бит), полуточность, используется для ускорения обучения огромных нейросетей (например, языковых моделей вроде GPT), чтобы экономить память видеокарты.

- **LongTensor** int64 (64 бита), целые числа, используется для меток классов (целевой переменной y). В вашей задаче массив y_true (где лежат только 0 и 1) дальше по коду превратят именно в LongTensor, потому что нейросеть требует, чтобы номера классов были строго целыми числами.

- **IntTensor** int32 (32 бита), обычные целые числа, используются для простых счетчиков или индексов.

Переносим данные из NumPy (numpy.ndarray) в PyTorch, в объект типа **torch.Tensor**, его можно мгновенно перекинуть на видеокарту (.cuda()) для сверхбыстрого обучения нейросети.

>Функция torch.FloatTensor принудительно пережимает каждое число из 64-битного формата (float64 по умолчанию для NumPy) в 32-битный формат **float32**.

Зачем это делается?
- Экономия памяти в 2 раза  

- Графические процессоры (GPU) выполняют операции с числами float32 аппаратно в несколько раз быстрее  

- Почти все стандартные слои в PyTorch ожидают именно float32

In [ ]:
# для PyTorch сделаем permute (Batch - окна, Channels - датчики, Length - замеры)
X_tensor = torch.FLoatTensor(X_windows).permute(0, 2, 1) #  4951, 1, 50

# Обучающая выборка (только нормальные окна)
X_train_tensor = X_tensor[:2000]

# Модель Conv1D - Одномерный сверточный автокодировщик
# (1, 50) -> (16, 13) -> (1, 50)

class Conv1DAE(nn.Module): # Наследуемся от nn.Module (веса, cuda, Backpropagation)
    def __init__(self):
        super().__init__() # Активация nn.Module

        # Sequential последовательно передает данные из одного слоя в другой
        self.encoder = nn.Sequential(
            nn.Conv1d(1,8,kernel_size=5, stride=2, padding=2), nn.ReLU(), # свертка 
            nn.Conv1d(8,16,kernel_size=5, stride=2, padding=2), nn.ReLU() # свертка 
        )

        # ConvTranspose1d транспонированные свертки (развёртки)
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(16, 8, kernel_size=5, stride=2, padding=2, output_padding=0), nn.ReLU(),
            nn.ConvTranspose1d(8, 1, kernel_size=5, stride=2, padding=2, output_padding=1)
        )

    # Прямой проход
    def forward(self, x):
        return self.decoder(self.encoder(x))

### 4. Тренер и судья

`MSELoss` расшифровывается как **Mean Squared Error Loss** (Среднеквадратичная ошибка). Это «судья», который будет оценивать, насколько хорошо нейросеть справилась с восстановлением сигнала.

Как она считает ошибку для нашего окна:

1. Нейросеть принимает реальное окно вибрации (X) и выдает свою восстановленную версию (X̂).  

2. `MSELoss` берет каждую из 50 точек, находит разницу между реальным числом и тем, что восстановила сеть, и **возводит эту разницу в квадрат** (чтобы и положительные, и отрицательные отклонения одинаково штрафовали модель).  

3. Затем она считает среднее арифметическое этих квадратов по всем 50 точкам.

**Почему именно MSE?** Из-за возведения в квадрат эта функция потерь очень сурово наказывает модель за _крупные_ ошибки. Если сеть сильно исказит форму сигнала, MSE мгновенно взлетит вверх, заставляя сеть переучиваться.


`Оптимизатор` — это алгоритм, который подкручивает внутренние веса нейросети, чтобы уменьшить ошибку, посчитанную `MSELoss`. **Adam** (Adaptive Moment Estimation) — на сегодняшний день это самый популярный, надежный и эффективный оптимизатор в ИИ.

- **`model.parameters()`** — мы передаем оптимизатору доступ ко всем внутренним весам (регулировочным винтам) нашей модели `Conv1DAE`. Он будет знать, какие именно параметры ему разрешено изменять.
- **`lr=0.01` (Learning Rate / Скорость обучения)** — это размер шага, с которым оптимизатор подкручивает веса.
    - `0.01` — это достаточно высокая скорость обучения (агрессивный шаг). Модель будет учиться быстро.
    - Если сделать шаг слишком большим (например, `0.1`), модель будет действовать слишком грубо и может испортить веса. Если сделать слишком маленьким (`0.0001`), обучение займет вечность.

In [ ]:
# Вызываем конструктор класса и создаем объект модели в оперативной памяти
model = Conv1DAE()

# Тренер (оптимизатор)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Cудья (Функция потерь)
criterion = nn.MSELoss()

### 5. Обучение

In [ ]:
model.train()
for epoch in range(50): # 50 эпох для скорости
    optimizer.zero_grad()
    loss = criterion(model(X_train_tensor), X_train_tensor)
    loss.backward()
    optimizer.step()